# NB1 — Data EDA + Panel Fingerprint (Abrigo Rev-2 migration)

**Purpose.** This notebook validates the Rev-5.3.2 panel construction; estimation lives in NB2 (`02_estimation.ipynb`); specification tests live in NB3 (`03_tests_and_sensitivity.ipynb`).

**Upstream input.** 14 Phase 5a panel parquets at `contracts/.scratch/2026-04-25-task110-rev2-data/` (`panel_row_01_primary.parquet` … `panel_row_14_wc_cpi_weights_sens.parquet`) plus the Phase 5a documentation (`manifest.md`, `data_dictionary.md`, `validation.md`, `_audit_summary.json`) and the Rev-5.3.2 published DuckDB state at `contracts/data/structural_econ.duckdb`.

**Downstream consumers.** NB2 consumes the panel-fingerprint JSON emitted by NB1 §1. NB3 consumes the Y₃ component decomposition emitted by NB1 §2 plus the outlier diagnostics emitted by NB1 §6.

**Pre-committed gate verdict.** FAIL (one-sided T3b). Reproduced byte-exact from `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` in NB2 / NB3; never re-estimated in this migration.

---

## §0 — Notebook header + panel-fingerprint validation

### Why-markdown (4-part citation block)

**Reference.**

- Rev-2 spec, autonomous track A, at `contracts/.scratch/2026-04-25-task110-rev2-spec-A-autonomous.md` (655 lines; §10.6 ζ-group roadmap; §11.A convex-payoff insufficiency caveat).
- Phase 5a Data Engineer outputs at `contracts/.scratch/2026-04-25-task110-rev2-data/` (14 panel parquets + `manifest.md` row-summary + `data_dictionary.md` + `validation.md` + `_audit_summary.json` machine-readable per-row audit).
- Phase 5b Analytics Reporter outputs at `contracts/.scratch/2026-04-25-task110-rev2-analysis/` (`estimates.md`, `spec_tests.md`, `sensitivity.md`, `summary.md`, `gate_verdict.json` with `gate_verdict = "FAIL"`).
- Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md` (the disposition that reclassifies the X_d source address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606` as Minteo-fintech, OUT of Mento-native scope).
- Mento-native address-registry spec at `contracts/docs/superpowers/specs/2026-04-25-mento-native-address-registry.md` (canonical address-registry; locks `0x8A567e2aE79CA692Bd748aB832081C45de4041eA` as Mento-native `StableTokenCOP`).
- Y₃ inequality-differential design at `contracts/docs/superpowers/specs/2026-04-24-y3-inequality-differential-design.md` (4-country panel CO/BR/KE/EU; equal-weight aggregation; pre-registered WC-CPI weights 60/25/15).
- X_d carbon-basket design at `contracts/docs/superpowers/specs/2026-04-24-carbon-basket-xd-design.md` (carbon-basket user-volume series).
- Project memory: `project_y3_inequality_differential_design`, `project_abrigo_inequality_hedge_thesis`, `project_abrigo_convex_instruments_inequality`, `project_abrigo_mento_native_only` (β-corrigendum).

**Why used.** Section 0 establishes the panel-fingerprint contract that pins every downstream cell to the Rev-5.3.2 published estimates. The fingerprint is the load-bearing acceptance criterion of the migration: NB1 / NB2 / NB3 reproduce the published numbers BYTE-EXACT, or the migration fails. The Phase 5a `_audit_summary.json` is the upstream authority for row counts, methodology tags, and date windows; the gate-verdict SHA pin is the downstream authority for the published verdict. By fingerprinting both at the entry point of NB1, any silent drift (library skew, parquet rewrite, seed mismatch, methodology-tag rename) surfaces here rather than propagating into NB2 or NB3.

**Relevance to results.** The published primary-row estimate (β̂_X_d = -2.7987e-8, HAC(4) SE = 1.4234e-8, t-stat = -1.9662, n = 76, window `[2024-09-27, 2026-03-13]`, gate verdict FAIL) is byte-exact-immutable per the Rev-5.3.x anti-fishing invariants (`MDES_FORMULATION_HASH = 4940360dcd2987...cefa`; `decision_hash = 6a5f9d1b05c1...443c`; `N_MIN = 75`; `POWER_MIN = 0.80`; `MDES_SD = 0.40`). §0 verifies the panel inputs that produced those estimates are themselves byte-exact: 14 panels with the published row counts (76 / 76 / 65 / 56 / 76 / 76 / 45 / 47 / 0 / 0 / 76 / 76 / 76 / 76); `source_methodology = y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` for the primary panel (Rev-5.3.2 default-flip post-Task 11.O Step-0, commit `202874565`); pre-registered FAIL rows at n = 65 (Row 3 LOCF-tail-excluded) and n = 56 (Row 4 IMF-only); deferred-empty Rows 9 and 10 with n = 0 per Phase 5a `manifest.md` §1.2 (spec §10 ε.2 / ε.3).

**Connection to product.** Abrigo sells convex (option-like) instruments that hedge macroeconomic shocks viewed through the inequality lens: the differential between rich-asset returns and working-class consumption returns. The Rev-2 mean-β estimation is first-stage / linear-hedge calibration only; per Rev-2 spec §11.A, mean-β is necessary-but-insufficient for convex-payoff product pricing. Convex-payoff fitness lives in Rev-3 ζ-group (quantile / GARCH-X / lower-tail / option-implied vol; deferred to Task 11.O.ζ-α per major-plan α-track scope). Crucially, Rev-2 measured X_d at the `0xc92e8fc2...` address; under the Rev-5.3.5 β-disposition that address is reclassified as Minteo-fintech (out of Mento-native scope), and the Mento-native COPm address `0x8A567e2a...` is the actual hedge-demand surface (deferred to Task 11.P.spec-β / Task 11.P.exec-β). Rev-2 therefore closes as scope-mismatch close-out on the Minteo-fintech X_d — not as a test of the Mento-native COPm hedge-demand surface.

In [1]:
"""NB1 §0 — panel-fingerprint validation.

Loads the 14 Phase 5a panels + the audit summary from
contracts/.scratch/2026-04-25-task110-rev2-data/, asserts byte-exact reproduction
of the published row counts / methodology tags / date windows, and emits the
gate-verdict SHA pin. Functional-Python style: frozen dataclasses, free pure
functions, full typing, no inheritance.
"""
from __future__ import annotations

import hashlib
import importlib.util
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import duckdb

# ---- Locate env.py and load by file path (notebooks/abrigo_y3_x_d/ is not on sys.path) ----

def _locate_abrigo_dir() -> Path:
    """Find the abrigo_y3_x_d/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    if (cwd / "env.py").is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / "contracts" / "notebooks" / "abrigo_y3_x_d"
        if (candidate / "env.py").is_file():
            return candidate
        candidate2 = parent / "notebooks" / "abrigo_y3_x_d"
        if (candidate2 / "env.py").is_file():
            return candidate2
    raise RuntimeError(f"Could not locate abrigo_y3_x_d/env.py starting from cwd={cwd}")


_ABRIGO_DIR: Path = _locate_abrigo_dir()
_ENV_PATH: Path = _ABRIGO_DIR / "env.py"
_spec = importlib.util.spec_from_file_location("abrigo_env", _ENV_PATH)
env = importlib.util.module_from_spec(_spec)
sys.modules["abrigo_env"] = env
_spec.loader.exec_module(env)

# ---- Pre-committed Phase 5a expected fingerprint (binding upstream contract) ----

PRIMARY_METHODOLOGY: str = "y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable"
PRIMARY_WINDOW: tuple[str, str] = ("2024-09-27", "2026-03-13")
PRIMARY_N: int = 76

EXPECTED_N_OBS: dict[str, int] = {
    "row_01_primary": 76,
    "row_02_bootstrap_recon": 76,
    "row_03_locf_tail_excluded": 65,
    "row_04_imf_only_sensitivity": 56,
    "row_05_lag_sensitivity": 76,
    "row_06_parsimonious_controls": 76,
    "row_07_arb_only": 45,
    "row_08_per_currency_copm": 47,
    "row_09_y3_bond_diagnostic": 0,
    "row_10_population_weighted": 0,
    "row_11_student_t": 76,
    "row_12_hac12_bandwidth": 76,
    "row_13_first_differenced": 76,
    "row_14_wc_cpi_weights_sens": 76,
}

DEFERRED_ROWS: frozenset[str] = frozenset({"row_09_y3_bond_diagnostic", "row_10_population_weighted"})


@dataclass(frozen=True, slots=True)
class PanelFingerprint:
    """Per-row panel fingerprint emitted from §0 (sanity-check structure)."""
    row_label: str
    parquet_path: Path
    n_obs: int
    dt_min: str | None
    dt_max: str | None
    y3_methodology: str | None
    deferred: bool


@dataclass(frozen=True, slots=True)
class FingerprintReport:
    """Aggregated panel-fingerprint report for §0 sanity check."""
    panels: tuple[PanelFingerprint, ...]
    gate_verdict_sha256: str
    gate_verdict_path: Path


# ---- Pure functions (no inheritance, no globals mutated) ----

def _phase5a_data_dir() -> Path:
    return env._CONTRACTS_DIR / ".scratch" / "2026-04-25-task110-rev2-data"


def _read_audit_summary(data_dir: Path) -> dict[str, dict]:
    with (data_dir / "_audit_summary.json").open() as fh:
        return json.load(fh)


def _query_panel(conn: duckdb.DuckDBPyConnection, parquet_path: Path) -> tuple[int, str | None, str | None]:
    if not parquet_path.is_file():
        raise FileNotFoundError(f"Phase 5a parquet missing: {parquet_path}")
    sql = f"SELECT COUNT(*) AS n, MIN(week_start) AS dt_min, MAX(week_start) AS dt_max FROM '{parquet_path}'"
    row = conn.sql(sql).fetchone()
    n = int(row[0])
    dt_min = str(row[1]) if row[1] is not None else None
    dt_max = str(row[2]) if row[2] is not None else None
    return n, dt_min, dt_max


def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def build_fingerprint_report(data_dir: Path, gate_verdict_path: Path) -> FingerprintReport:
    audit = _read_audit_summary(data_dir)
    conn = duckdb.connect()
    panels: list[PanelFingerprint] = []
    for row_label, expected_n in EXPECTED_N_OBS.items():
        parquet_path = data_dir / f"panel_{row_label}.parquet"
        n_obs, dt_min, dt_max = _query_panel(conn, parquet_path)
        is_deferred = row_label in DEFERRED_ROWS
        audit_row = audit[row_label]
        # Byte-exact assertions vs. _audit_summary.json + EXPECTED_N_OBS
        assert n_obs == expected_n, f"{row_label}: parquet n={n_obs} != EXPECTED_N_OBS={expected_n}"
        assert n_obs == int(audit_row["n_obs"]), f"{row_label}: parquet n={n_obs} != audit n_obs={audit_row['n_obs']}"
        if not is_deferred:
            assert dt_min == audit_row["dt_min"], f"{row_label}: dt_min mismatch {dt_min} vs {audit_row['dt_min']}"
            assert dt_max == audit_row["dt_max"], f"{row_label}: dt_max mismatch {dt_max} vs {audit_row['dt_max']}"
        panels.append(
            PanelFingerprint(
                row_label=row_label,
                parquet_path=parquet_path,
                n_obs=n_obs,
                dt_min=dt_min,
                dt_max=dt_max,
                y3_methodology=audit_row.get("y3_methodology"),
                deferred=is_deferred,
            )
        )
    conn.close()
    sha = _sha256_file(gate_verdict_path)
    return FingerprintReport(panels=tuple(panels), gate_verdict_sha256=sha, gate_verdict_path=gate_verdict_path)


# ---- Execute the fingerprint validation ----

_DATA_DIR = _phase5a_data_dir()
report = build_fingerprint_report(_DATA_DIR, env.GATE_VERDICT_PATH)

# Primary-panel anchor assertions (the load-bearing claims of §0)
_primary = next(p for p in report.panels if p.row_label == "row_01_primary")
assert _primary.n_obs == PRIMARY_N, f"Primary n_obs drift: got {_primary.n_obs}, expected {PRIMARY_N}"
assert _primary.y3_methodology == PRIMARY_METHODOLOGY, (
    f"Primary methodology drift: got {_primary.y3_methodology!r}, expected {PRIMARY_METHODOLOGY!r}"
)
assert (_primary.dt_min, _primary.dt_max) == PRIMARY_WINDOW, (
    f"Primary window drift: got ({_primary.dt_min}, {_primary.dt_max}), expected {PRIMARY_WINDOW}"
)

# Pre-registered FAIL anchors (Row 3 LOCF-tail-excluded; Row 4 IMF-only)
_row3 = next(p for p in report.panels if p.row_label == "row_03_locf_tail_excluded")
_row4 = next(p for p in report.panels if p.row_label == "row_04_imf_only_sensitivity")
assert _row3.n_obs == 65, f"Row 3 pre-registered FAIL anchor drift: n={_row3.n_obs}, expected 65"
assert _row4.n_obs == 56, f"Row 4 pre-registered FAIL anchor drift: n={_row4.n_obs}, expected 56"

# Deferred-empty anchors (Rows 9, 10 per Phase 5a manifest §1.2 / spec §10 ε.2/ε.3)
for _label in ("row_09_y3_bond_diagnostic", "row_10_population_weighted"):
    _row = next(p for p in report.panels if p.row_label == _label)
    assert _row.n_obs == 0, f"{_label} deferred-empty anchor drift: n={_row.n_obs}, expected 0"
    assert _row.deferred, f"{_label} not flagged deferred"

# Emit summary table inline (sanity check; downstream JSON emission is sub-task 2 / NB1 §1)
print(f"Phase 5a data dir: {_DATA_DIR}")
print(f"gate_verdict.json SHA-256: {report.gate_verdict_sha256}")
print(f"gate_verdict.json path: {report.gate_verdict_path}")
print(f"Primary methodology: {PRIMARY_METHODOLOGY}")
print(f"Primary window: [{PRIMARY_WINDOW[0]}, {PRIMARY_WINDOW[1]}]  n={PRIMARY_N}")
print()
print(f"{'row_label':<32} {'n_obs':>6} {'dt_min':<12} {'dt_max':<12} deferred")
for p in report.panels:
    print(
        f"{p.row_label:<32} {p.n_obs:>6} "
        f"{(p.dt_min or '-'):<12} {(p.dt_max or '-'):<12} "
        f"{p.deferred}"
    )

Phase 5a data dir: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/.scratch/2026-04-25-task110-rev2-data
gate_verdict.json SHA-256: 29f716ec835bb693f8985aeea9c97215aaf804931e916dbbf5afdf0cdf6e0259
gate_verdict.json path: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/notebooks/abrigo_y3_x_d/estimates/gate_verdict.json
Primary methodology: y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable
Primary window: [2024-09-27, 2026-03-13]  n=76

row_label                         n_obs dt_min       dt_max       deferred
row_01_primary                       76 2024-09-27   2026-03-13   False
row_02_bootstrap_recon               76 2024-09-27   2026-03-13   False
row_03_locf_tail_excluded            65 2024-09-27   2025-12-26   False
row_04_imf_only_sensitivity          56 2024-09-27   2025-10-24   False
row_05_lag_sensitivity               76 2024-09-27   2026-03-13   False
row_06_parsimonious_controls         76 

### Interpretation

**Panel-fingerprint validation establishes the byte-exact upstream contract for the migration.** All 14 Phase 5a panels load from `contracts/.scratch/2026-04-25-task110-rev2-data/` with row counts matching `_audit_summary.json` exactly: the primary panel at n = 76 over the window `[2024-09-27, 2026-03-13]` under `source_methodology = y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` (Rev-5.3.2 default-flip post-Task 11.O Step-0); the two pre-registered FAIL anchors at n = 65 (Row 3 LOCF-tail-excluded) and n = 56 (Row 4 IMF-only); and the two deferred-empty rows at n = 0 (Row 9 Y₃-bond diagnostic, Row 10 population-weighted Y₃ — both deferred per spec §10 ε.2 / ε.3). The gate-verdict SHA-256 is emitted and the file is pinned at `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` so any future drift to the published verdict (gate_verdict = FAIL; β̂_X_d = -2.7987e-8; HAC(4) SE = 1.4234e-8; n = 76) would surface here rather than propagating silently into NB2 / NB3.

**Scope framing under Rev-5.3.5.** This notebook re-presents the Rev-2 panels, which used the Minteo-fintech X_d source at address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606`. Per the Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md`, that address is now classified as Minteo-fintech (out of Mento-native scope per `project_abrigo_mento_native_only`); the canonical Mento-native `StableTokenCOP` address is `0x8A567e2aE79CA692Bd748aB832081C45de4041eA` per the address-registry spec at `contracts/docs/superpowers/specs/2026-04-25-mento-native-address-registry.md`. Rev-2 therefore closes as **Minteo-fintech scope-mismatch close-out** on the byte-exact-immutable estimates: the published mean-β numbers stay in the audit trail unchanged, but the interpretation is **scope-mismatch close-out** on the Minteo-fintech X_d. Concretely, Rev-2 closes scope-mismatch on the Minteo-fintech X_d at `0xc92e8fc2...`; the Mento-native COPm X_d hypothesis (against `0x8A567e2a...`) is the actual surface of interest and is deferred to Task 11.P.spec-β / Task 11.P.exec-β (the β-track Rev-3 sub-plan).

**Convex-payoff caveat preserved (Rev-2 spec §11.A).** Even setting aside the Minteo-fintech scope-mismatch close-out, mean-β identification is *necessary-but-insufficient* for convex-payoff product pricing. Whatever T3b yields on a future Mento-native panel is first-stage / linear-hedge calibration only; convex-payoff fitness (option / cap / floor instruments) requires the Rev-3 ζ-group extensions — quantile regression, GARCH-X, lower-tail dependence, option-implied-vol triangulation — which are deferred to Task 11.O.ζ-α per the major-plan α-track scope. NB3 §5 reproduces the spec §11.A caveat verbatim downstream; §0 here flags it because the panel-fingerprint contract does not by itself bound convex-payoff hedge fitness. Cross-references: `project_abrigo_convex_instruments_inequality`, `project_abrigo_inequality_hedge_thesis`, `project_y3_inequality_differential_design`, `project_abrigo_mento_native_only`.

---

## §1 — Per-row diagnostics (14-row panel structure validation)

### Why-markdown (4-part citation block)

**Reference.**

- Phase 5a Data Engineer outputs at `contracts/.scratch/2026-04-25-task110-rev2-data/`: the 14 `panel_row_*.parquet` artifacts plus `_audit_summary.json` (machine-readable per-row audit), `manifest.md` (row-level summary table — Analytics Reporter input contract per §6), `data_dictionary.md` (variable definitions, units, cleaning steps), and `validation.md` (per-row data-quality checks).
- Rev-2 spec §10 14-row resolution matrix at `contracts/.scratch/2026-04-25-task110-rev2-spec-A-autonomous.md`: the published row enumeration (Row 1 primary; Rows 3 + 4 pre-registered FAIL anchors; Rows 9 + 10 deferred per ε.2 / ε.3; Rows 2, 5–8, 11–14 OPEN).
- Memory `feedback_notebook_citation_block` (NON-NEGOTIABLE): every test / decision / spec choice in an estimation or sensitivity notebook is preceded by a 4-part citation block (reference / why used / relevance to results / connection to simulator).

**Why used.** The per-row diagnostic table is the falsifiable acceptance gate at the panel-structure layer of the migration. §0 fingerprinted the upstream contract at the gate-bearing summary level (n_obs / dt_min / dt_max / methodology tag for each row). §1 expands that fingerprint to the per-row column-level structure: column count, non-null fraction per column, methodology tag, and date span. Pinning the panel structure byte-exact against Phase 5a forces any drift (parquet rewrite, schema rename, column-set change, null-handling regression) to surface here rather than silently propagating into NB2 estimation. The `_audit_summary.json` is the upstream authority for n_obs / dt_min / dt_max / y3_methodology; the parquet files themselves are the authority for column-set and per-column non-null fractions.

**Relevance to results.** Byte-exact panel-fingerprint reproduction is the necessary condition for byte-exact estimate reproduction in NB2. The Rev-5.3.2 published estimates (β̂_X_d = -2.7987e-8, HAC(4) SE = 1.4234e-8, t-stat = -1.9662, n = 76, gate verdict FAIL) are anti-fishing-immutable per the Rev-5.3.x invariants (`MDES_FORMULATION_HASH`, `decision_hash`, `N_MIN`, `POWER_MIN`, `MDES_SD`); any per-row column-set drift would invalidate the regression-input contract that produced those numbers. The per-row diagnostic table emitted in this section is therefore the primary defense against silent input-drift between Phase 5a (panel construction) and NB2 (estimation re-presentation).

**Connection to simulator.** The per-row diagnostic table feeds the simulator's input-data validation hook: at simulator-load time the simulator reads the panel-fingerprint JSON (emitted by §1 trio 2 — cross-row consistency aggregation) and asserts byte-exact reproduction of n_obs, column count, and methodology tag for every row before any pricing computation runs. Any drift triggers a HALT-VERIFY gate, blocking the simulator from consuming a panel structurally inconsistent with the published estimates. The Rev-2 panels close as **Minteo-fintech scope-mismatch close-out** under the Rev-5.3.5 β-disposition: the byte-exact per-row diagnostic stays in the audit trail, but the actual Mento-native COPm X_d hypothesis (against `0x8A567e2a...`) is deferred to Task 11.P.spec-β / Task 11.P.exec-β. Concretely: Rev-2 closes scope-mismatch on the Minteo-fintech X_d at `0xc92e8fc2...`, and the per-row diagnostic emitted here is the input contract that the β-track Mento-native panel must replace cell-for-cell when it lands.

In [2]:
"""NB1 §1 trio 1 — per-row diagnostics.

For each of the 14 Phase 5a panel parquets, computes the per-row diagnostic
record: n_obs, column count, non-null fraction per column (dict + summary),
date-coverage span, source methodology tag. Renders a 14-row diagnostic
DataFrame; asserts byte-exact reproduction of the upstream `_audit_summary.json`
n_obs / dt_min / dt_max / y3_methodology fields. Functional-Python style:
frozen dataclasses, free pure functions, full typing, no inheritance.

Trio 2 (cross-row consistency aggregation + panel-fingerprint JSON emission)
follows in a separate dispatch.
"""
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from types import MappingProxyType
from typing import Mapping

import duckdb
import pandas as pd

# ---- Frozen-dataclass per-row diagnostic record ----

@dataclass(frozen=True, slots=True)
class PerRowDiagnostic:
    """Per-row diagnostic for a single Phase 5a panel parquet."""
    panel_id: str
    n_obs: int
    n_cols: int
    column_set: tuple[str, ...]
    non_null_fraction: Mapping[str, float]
    date_min: str | None
    date_max: str | None
    source_methodology: str | None
    deferred: bool


# ---- Pure helper functions ----

def _audit_path(data_dir: Path) -> Path:
    return data_dir / "_audit_summary.json"


def _load_audit(data_dir: Path) -> Mapping[str, Mapping[str, object]]:
    with _audit_path(data_dir).open() as fh:
        return json.load(fh)


def _panel_path(data_dir: Path, panel_id: str) -> Path:
    return data_dir / f"panel_{panel_id}.parquet"


def _query_columns_and_nullfrac(
    conn: duckdb.DuckDBPyConnection,
    parquet_path: Path,
    n_obs: int,
) -> tuple[tuple[str, ...], Mapping[str, float]]:
    """Return (ordered column tuple, per-column non-null fraction)."""
    if n_obs == 0:
        # Deferred-empty panels: introspect schema only.
        cols_df = conn.sql(
            f"DESCRIBE SELECT * FROM '{parquet_path}'"
        ).df()
        cols = tuple(str(c) for c in cols_df["column_name"].tolist())
        return cols, MappingProxyType({c: 0.0 for c in cols})
    cols_df = conn.sql(
        f"DESCRIBE SELECT * FROM '{parquet_path}'"
    ).df()
    cols = tuple(str(c) for c in cols_df["column_name"].tolist())
    select_clauses = ", ".join(
        f"SUM(CASE WHEN \"{c}\" IS NOT NULL THEN 1 ELSE 0 END)::DOUBLE / "
        f"{n_obs}::DOUBLE AS \"{c}\""
        for c in cols
    )
    nf_row = conn.sql(
        f"SELECT {select_clauses} FROM '{parquet_path}'"
    ).fetchone()
    non_null = {c: float(nf_row[i]) for i, c in enumerate(cols)}
    return cols, MappingProxyType(non_null)


def _query_date_span(
    conn: duckdb.DuckDBPyConnection,
    parquet_path: Path,
    n_obs: int,
) -> tuple[str | None, str | None]:
    if n_obs == 0:
        return None, None
    row = conn.sql(
        f"SELECT MIN(week_start), MAX(week_start) FROM '{parquet_path}'"
    ).fetchone()
    dt_min = str(row[0]) if row[0] is not None else None
    dt_max = str(row[1]) if row[1] is not None else None
    return dt_min, dt_max


def diagnose_panel(
    conn: duckdb.DuckDBPyConnection,
    data_dir: Path,
    panel_id: str,
    audit_row: Mapping[str, object],
) -> PerRowDiagnostic:
    parquet_path = _panel_path(data_dir, panel_id)
    if not parquet_path.is_file():
        raise FileNotFoundError(f"Phase 5a parquet missing: {parquet_path}")
    audit_n_obs = int(audit_row["n_obs"])  # type: ignore[arg-type]
    n_obs_row = conn.sql(
        f"SELECT COUNT(*) FROM '{parquet_path}'"
    ).fetchone()
    n_obs = int(n_obs_row[0])
    assert n_obs == audit_n_obs, (
        f"{panel_id}: parquet n_obs={n_obs} != audit n_obs={audit_n_obs}"
    )
    cols, nullfrac = _query_columns_and_nullfrac(conn, parquet_path, n_obs)
    dt_min, dt_max = _query_date_span(conn, parquet_path, n_obs)
    if n_obs > 0:
        assert dt_min == audit_row.get("dt_min"), (
            f"{panel_id}: dt_min mismatch {dt_min} vs {audit_row.get('dt_min')}"
        )
        assert dt_max == audit_row.get("dt_max"), (
            f"{panel_id}: dt_max mismatch {dt_max} vs {audit_row.get('dt_max')}"
        )
    deferred = bool(audit_row.get("deferred", False))
    methodology = audit_row.get("y3_methodology")
    return PerRowDiagnostic(
        panel_id=panel_id,
        n_obs=n_obs,
        n_cols=len(cols),
        column_set=cols,
        non_null_fraction=nullfrac,
        date_min=dt_min,
        date_max=dt_max,
        source_methodology=methodology if methodology is None else str(methodology),
        deferred=deferred,
    )


def diagnose_all_panels(
    data_dir: Path,
    panel_order: tuple[str, ...],
) -> tuple[PerRowDiagnostic, ...]:
    audit = _load_audit(data_dir)
    conn = duckdb.connect()
    try:
        out: list[PerRowDiagnostic] = []
        for panel_id in panel_order:
            assert panel_id in audit, (
                f"{panel_id} missing from _audit_summary.json"
            )
            out.append(diagnose_panel(conn, data_dir, panel_id, audit[panel_id]))
        return tuple(out)
    finally:
        conn.close()


def diagnostics_to_frame(
    diagnostics: tuple[PerRowDiagnostic, ...],
) -> pd.DataFrame:
    rows = []
    for d in diagnostics:
        if d.n_obs == 0:
            n_with_nulls = 0
            max_null_frac = 0.0
        else:
            n_with_nulls = sum(
                1 for f in d.non_null_fraction.values() if f < 1.0
            )
            max_null_frac = (
                max(1.0 - f for f in d.non_null_fraction.values())
                if d.non_null_fraction
                else 0.0
            )
        rows.append(
            {
                "panel_id": d.panel_id,
                "n_obs": d.n_obs,
                "n_cols": d.n_cols,
                "date_min": d.date_min if d.date_min is not None else "-",
                "date_max": d.date_max if d.date_max is not None else "-",
                "source_methodology": (
                    d.source_methodology
                    if d.source_methodology is not None
                    else "-"
                ),
                "n_columns_with_nulls": n_with_nulls,
                "max_null_fraction": round(max_null_frac, 6),
                "deferred": d.deferred,
            }
        )
    return pd.DataFrame(rows)


# ---- Pre-committed canonical 14-row order (Phase 5a manifest §1.2) ----

PANEL_ORDER: tuple[str, ...] = (
    "row_01_primary",
    "row_02_bootstrap_recon",
    "row_03_locf_tail_excluded",
    "row_04_imf_only_sensitivity",
    "row_05_lag_sensitivity",
    "row_06_parsimonious_controls",
    "row_07_arb_only",
    "row_08_per_currency_copm",
    "row_09_y3_bond_diagnostic",
    "row_10_population_weighted",
    "row_11_student_t",
    "row_12_hac12_bandwidth",
    "row_13_first_differenced",
    "row_14_wc_cpi_weights_sens",
)


# ---- Execute ----

_DATA_DIR_NB1_S1 = _DATA_DIR  # reuse §0 path; same Phase 5a authority
_DIAGNOSTICS = diagnose_all_panels(_DATA_DIR_NB1_S1, PANEL_ORDER)
assert len(_DIAGNOSTICS) == 14, f"Expected 14 diagnostics, got {len(_DIAGNOSTICS)}"

# Primary-panel structural anchor: 6 controls + (week_start, y3_value, x_d) +
# 4 per-currency diff columns => column-set must include the named regressors.
_primary_diag = _DIAGNOSTICS[0]
assert _primary_diag.panel_id == "row_01_primary"
assert _primary_diag.n_obs == 76
for required_col in (
    "week_start",
    "y3_value",
    "x_d",
    "vix_avg",
    "oil_return",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "fed_funds_weekly",
    "intervention_dummy",
):
    assert required_col in _primary_diag.column_set, (
        f"row_01_primary missing required column: {required_col}"
    )

# Render the 14-row per-row diagnostic frame inline.
_DIAG_FRAME = diagnostics_to_frame(_DIAGNOSTICS)
with pd.option_context(
    "display.max_columns", None,
    "display.width", 200,
    "display.max_colwidth", 80,
):
    print(_DIAG_FRAME.to_string(index=False))


                    panel_id  n_obs  n_cols   date_min   date_max                                               source_methodology  n_columns_with_nulls  max_null_fraction  deferred
              row_01_primary     76      13 2024-09-27 2026-03-13 y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable                     1                1.0     False
      row_02_bootstrap_recon     76      13 2024-09-27 2026-03-13 y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable                     1                1.0     False
   row_03_locf_tail_excluded     65      13 2024-09-27 2025-12-26 y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable                     1                1.0     False
 row_04_imf_only_sensitivity     56      13 2024-09-27 2025-10-24               y3_v2_imf_only_sensitivity_3country_ke_unavailable                     1                1.0     False
      row_05_lag_sensitivity     76      13 2024-09-27 2026-03-13 y3_v2_co_dane_br_bcb_eu_

### Interpretation

**Per-row diagnostics establish byte-exact panel-structure reproduction across all 14 Rev-2 panels.** Every row in the diagnostic table reproduces its `_audit_summary.json` n_obs / dt_min / dt_max / y3_methodology fields exactly: the gate-bearing primary at n = 76 over `[2024-09-27, 2026-03-13]`; Row 2 bootstrap-reconciliation at n = 76 (same panel-set as Row 1, estimator differs); the two pre-registered FAIL anchors at n = 65 (Row 3 LOCF-tail-excluded; window truncated at `2025-12-26`) and n = 56 (Row 4 IMF-only sensitivity; window truncated at `2025-10-24`); the diagnostic under-N rows at n = 45 (Row 7 arb-only) and n = 47 (Row 8 per-currency COPM); the four refit-at-fit-time rows at n = 76 (Row 5 lag, Row 11 Student-t, Row 12 HAC(12), Row 13 first-differenced); the parsimonious-controls row at n = 76 (Row 6 — three controls only: `vix_avg`, `oil_return`, `intervention_dummy`); and the WC-CPI-weights-sensitivity row at n = 76 (Row 14 — same row-set as Row 1, alt-weight aggregation at fit time). The primary panel column-set passes the structural assertion that all six published regressors plus `week_start`, `y3_value`, and `x_d` are present.

**Source methodology aligns with the Rev-5.3.2 default-flip.** The primary panel and Rows 2, 3, 5–8, 11–14 carry `y3_methodology = y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` — the published Rev-5.3.2 default-flip literal post-Task 11.O Step-0 (commit `202874565`). Row 4 carries the IMF-IFS-only sensitivity tag `y3_v2_imf_only_sensitivity_3country_ke_unavailable` per spec §10. Rows 9 and 10 carry no methodology tag because they are deferred-empty per Phase 5a `manifest.md` §1.2 (spec §10 ε.2 / ε.3): the Y₃-bond rich-side construct (Row 9) requires a bond-data fetcher unbuilt in DuckDB, and the population-weighted Y₃ aggregator (Row 10) requires extending `y3_compute` with a weight-vector argument — both are out-of-scope per spec §14. The empty schema-typed parquets honor the 14-row deliverable contract; the diagnostic table flags them as `deferred = True`.

**Date-coverage spans align with the pre-registered Rev-2 spec windowing.** The four LOCF-window-aware rows (3, 4, 7, 8) each carry an early-cutoff `dt_max` per their spec methodology (Row 3: `2025-12-26` from the LOCF-tail-excluded cutoff `2025-12-31`; Row 4: `2025-10-24` from the IMF-only data-availability boundary; Row 7: `2025-07-04` from the BancorArbitrage trader-filter availability; Row 8: `2025-09-26` from the per-currency COPM availability), and the ten full-window rows all carry `dt_max = 2026-03-13` matching the Rev-2 cutoff.

**One by-design 100%-null column per non-deferred panel: `kes_diff` (Kenya).** The diagnostic table reports `n_columns_with_nulls = 1` and `max_null_fraction = 1.0` for every non-deferred row. Inspection identifies the single null-bearing column as `kes_diff` (the Kenya per-currency working-class CPI differential), which is uniformly null because the methodology tag itself encodes the skip: `..._ke_skip_3country_ke_unavailable` declares that Kenya WC-CPI data is unavailable and the country is excluded from the equal-weight aggregation. The `kes_diff` column persists in the parquet schema for column-set uniformity across the 14 panels (so column-position assertions hold cell-for-cell), but no row carries a value. This is by-design pre-registered behavior, not data drift. The `_audit_summary.json` field `n_null_y3 = 0` is corroborated cell-for-cell: the **outcome variable** `y3_value` carries no nulls in any non-deferred panel, and the only fully-null column is the diagnostic per-country differential whose absence is encoded in the methodology tag itself. The deferred Rows 9 + 10 carry the empty schema-typed contract (`n_obs = 0`, all columns trivially "non-null" by vacuous truth, `n_columns_with_nulls = 0`, `max_null_fraction = 0.0`).

**Scope framing under Rev-5.3.5.** The byte-exact per-row diagnostic preserves the audit trail of the Rev-2 panels, which used the Minteo-fintech X_d source at `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606`. Per the Rev-5.3.5 β-disposition memo, Rev-2 closes scope-mismatch on the Minteo-fintech X_d (out of Mento-native scope per `project_abrigo_mento_native_only`); the Mento-native COPm X_d hypothesis (against `0x8A567e2aE79CA692Bd748aB832081C45de4041eA`) is the actual surface of interest and is deferred to Task 11.P.spec-β / Task 11.P.exec-β. The per-row diagnostic emitted here is the structural input contract that the β-track Mento-native panel must replace cell-for-cell — the column-set (including the `kes_diff` schema-uniformity placeholder), methodology-tag conventions, and date-window discipline carry forward unchanged into the β-track Rev-3 sub-plan.

**Forward-pointer.** Trio 2 (cross-row consistency aggregation + panel-fingerprint JSON emission) follows. It aggregates this 14-row per-row diagnostic into a panel-level fingerprint (column-set hash per row + per-row n_obs vector + methodology-tag list + date-coverage span list), asserts that the 14 panels differ ONLY by their published spec-row methodology (column-set is byte-identical for the gate-bearing rows; deferred Rows 9 + 10 carry the empty schema-typed contract), and emits the JSON to `notebooks/abrigo_y3_x_d/estimates/panel_fingerprint.json` for downstream consumption by NB2 §1 and NB3.

### §1.b — Cross-row consistency aggregation + panel-fingerprint JSON emission (trio 2)

#### Why-markdown (4-part citation block)

**Reference.**

- Phase 5a `_audit_summary.json` at `contracts/.scratch/2026-04-25-task110-rev2-data/_audit_summary.json` — the upstream byte-exact authority for `n_obs`, `dt_min`, `dt_max`, `y3_methodology`, `controls`, and `x_d_kind` per row.
- Rev-2 spec §10 14-row resolution matrix at `contracts/.scratch/2026-04-25-task110-rev2-spec-A-autonomous.md` (lines reproduced cell-for-cell in §0 Reference block) — published row enumeration; deferred Rows 9 + 10 per ε.2 / ε.3.
- Trio 1 output (this notebook §1) — 14 `PerRowDiagnostic` records with column-set, n_obs, n_cols, non-null fractions, date span, and methodology tag.
- NB-α sub-plan §C Block A sub-task 2 at `contracts/docs/superpowers/sub-plans/2026-04-25-rev2-notebook-migration.md` — the dispatch contract for trio 2: cross-row consistency check + panel-fingerprint JSON emission.
- FX-vol-CPI Colombia precedent at `contracts/notebooks/fx_vol_cpi_surprise/Colombia/estimates/nb1_panel_fingerprint.json` — schema-continuity reference for fingerprint JSON style (decision-ledger + per-panel sha256 + column-dtype dict).
- Project memory: `feedback_notebook_citation_block`, `feedback_notebook_trio_checkpoint` (NON-NEGOTIABLE single-trio HALT discipline).

**Why used.** Cross-row consistency is the second falsifiability gate beyond per-row byte-exact reproduction. Trio 1 (NB1 §1) verifies that EACH of the 14 panels reproduces its `_audit_summary.json` row exactly. Trio 2 verifies that the 14 panels are STRUCTURALLY interrelated as published — i.e., pairs of non-deferred rows must share an identical column set (because every non-deferred row carries the same `(week_start, y3_value, x_d, 6 controls or parsimonious-3 controls, 4 per-currency diff columns)` schema modulo the published spec-row methodology), and the deferred-empty Rows 9 and 10 must declare their emptiness explicitly via `n_obs = 0` and a separate column-set comparison rule (not a structural mismatch). Drift between rows would silently corrupt the 14-row sensitivity ladder consumed by NB3, because NB3 fits per-row regressions assuming column-position uniformity. The panel-fingerprint JSON is the load-bearing input artifact for NB2 §1 (loads the primary panel for OLS+HAC(4)) and NB3 (loads each sensitivity-row panel); it locks the byte-exact contract so any drift surfaces at consumer-load time rather than silently propagating into estimates.

**Relevance to results.** The published Rev-5.3.2 estimates (β̂_X_d = -2.7987e-8, HAC(4) SE = 1.4234e-8, t-stat = -1.9662, n = 76, gate verdict FAIL) are anti-fishing-immutable per the Rev-5.3.x invariants. Cross-row consistency aggregation guarantees that the 14-row sensitivity ladder (Rows 1–14) was estimated on a structurally uniform panel set: Row 5 lag-sensitivity, Row 6 parsimonious-controls, Row 11 Student-t, Row 12 HAC(12), Row 13 first-differenced, and Row 14 WC-CPI-weights-sensitivity all share Row 1's primary column-set (so refit-at-fit-time transformations operate on the same input columns); Row 6 is intentionally a column-set subset (parsimonious controls drop `us_cpi_surprise`, `banrep_rate_surprise`, `fed_funds_weekly`); Row 4 IMF-only carries a different methodology tag but the same column-set; Rows 9 + 10 declare emptiness explicitly. Any deviation from this structural pattern would invalidate the 14-row sensitivity-ladder interpretation. The emitted `panel_fingerprint.json` byte-exact reproduces the `_audit_summary.json` overlap fields (n_rows_per_panel, date_min_per_panel, date_max_per_panel) so any drift between trio 2 and the upstream Phase 5a authority surfaces here as a hard assertion failure.

**Connection to product.** The `panel_fingerprint.json` becomes the simulator's input-data manifest: when the Abrigo simulator loads a panel for instrument-pricing computation, it asserts byte-exact reproduction of the fingerprint before any computation runs — n_rows, column count, non-null fractions, date-coverage span, and cross-row-consistency flag are all gated. Future Rev-3 ζ-group panels (quantile / GARCH-X / lower-tail / option-implied vol) will append under the same schema, providing a single source-of-truth for analytical-stage data lineage across mean-β (Rev-2) and convex-payoff (Rev-3) extensions. Crucially, under the Rev-5.3.5 β-disposition the Rev-2 panels close as **Minteo-fintech scope-mismatch close-out** on the Minteo-fintech X_d at `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606`; the Mento-native COPm X_d hypothesis (against `0x8A567e2aE79CA692Bd748aB832081C45de4041eA`) is deferred to Task 11.P.spec-β / Task 11.P.exec-β. The fingerprint emitted here is the structural input contract that the β-track Mento-native panel must replace cell-for-cell — the `n_rows_per_panel`, `column_count_per_panel`, and `non_null_fraction_per_column` envelopes carry forward unchanged. Concretely, **Rev-2 closes scope-mismatch on the Minteo-fintech X_d**; this fingerprint is the input-manifest template for the upcoming β-track Mento-native panel.

In [3]:
"""NB1 §1.b trio 2 — cross-row consistency aggregation + panel-fingerprint JSON emission.

Aggregates the 14 PerRowDiagnostic records from §1 trio 1 into a panel-level
fingerprint with the 6 required fields (n_rows_per_panel, column_count_per_panel,
non_null_fraction_per_column, date_min_per_panel, date_max_per_panel,
cross_row_consistency_flag), asserts byte-exact reproduction against the upstream
Phase 5a `_audit_summary.json` for the overlap fields (n_obs / dt_min / dt_max),
and emits the JSON to `notebooks/abrigo_y3_x_d/estimates/panel_fingerprint.json`
for downstream consumption by NB2 §1 + NB3.

Functional-Python style: frozen dataclasses, free pure functions, full typing,
no inheritance. Migration / re-presentation only — no re-estimation, no DuckDB
row mutations.
"""
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping


# ---- Frozen-dataclass aggregate fingerprint record ----

@dataclass(frozen=True, slots=True)
class CrossRowConsistencyPair:
    """Pairwise structural-consistency result for two panel rows."""
    panel_a: str
    panel_b: str
    column_set_equal: bool | None  # None => N/A (one or both panels deferred-empty)
    n_cols_a: int
    n_cols_b: int
    note: str


@dataclass(frozen=True, slots=True)
class PanelFingerprint:
    """Aggregated panel-fingerprint record for the 14 Rev-2 panels."""
    n_rows_per_panel: Mapping[str, int]
    column_count_per_panel: Mapping[str, int]
    non_null_fraction_per_column: Mapping[str, Mapping[str, float]]
    date_min_per_panel: Mapping[str, str | None]
    date_max_per_panel: Mapping[str, str | None]
    cross_row_consistency_flag: tuple[CrossRowConsistencyPair, ...]


# ---- Pure aggregation functions ----

def _aggregate_n_rows(diagnostics: tuple[PerRowDiagnostic, ...]) -> dict[str, int]:
    return {d.panel_id: d.n_obs for d in diagnostics}


def _aggregate_column_count(diagnostics: tuple[PerRowDiagnostic, ...]) -> dict[str, int]:
    return {d.panel_id: d.n_cols for d in diagnostics}


def _aggregate_non_null_fraction(
    diagnostics: tuple[PerRowDiagnostic, ...],
) -> dict[str, dict[str, float]]:
    # Convert MappingProxyType to plain dict for JSON serialization.
    return {d.panel_id: dict(d.non_null_fraction) for d in diagnostics}


def _aggregate_date_min(diagnostics: tuple[PerRowDiagnostic, ...]) -> dict[str, str | None]:
    return {d.panel_id: d.date_min for d in diagnostics}


def _aggregate_date_max(diagnostics: tuple[PerRowDiagnostic, ...]) -> dict[str, str | None]:
    return {d.panel_id: d.date_max for d in diagnostics}


def _compute_cross_row_consistency(
    diagnostics: tuple[PerRowDiagnostic, ...],
) -> tuple[CrossRowConsistencyPair, ...]:
    """For every ordered pair (i, j) with i < j, compute structural-consistency flag.

    Rule: column_set_equal is True iff frozenset(column_set_i) == frozenset(column_set_j)
    AND neither panel is deferred-empty. If either panel is deferred-empty (Rows 9 / 10),
    column_set_equal = None and the note records the deferred-empty rationale per Phase 5a
    manifest §1.2 (spec §10 ε.2 / ε.3).
    """
    pairs: list[CrossRowConsistencyPair] = []
    n = len(diagnostics)
    for i in range(n):
        for j in range(i + 1, n):
            di = diagnostics[i]
            dj = diagnostics[j]
            if di.deferred or dj.deferred:
                column_set_equal: bool | None = None
                if di.deferred and dj.deferred:
                    note = "BOTH_DEFERRED_EMPTY (spec §10 ε.2 + ε.3); column-set comparison N/A"
                elif di.deferred:
                    note = f"{di.panel_id} DEFERRED_EMPTY; column-set comparison N/A"
                else:
                    note = f"{dj.panel_id} DEFERRED_EMPTY; column-set comparison N/A"
            else:
                column_set_equal = frozenset(di.column_set) == frozenset(dj.column_set)
                if column_set_equal:
                    note = "column_sets identical (structural uniformity)"
                else:
                    sym_diff = sorted(set(di.column_set).symmetric_difference(set(dj.column_set)))
                    note = (
                        f"column_sets differ by spec-row methodology; "
                        f"symmetric_difference = {sym_diff}"
                    )
            pairs.append(
                CrossRowConsistencyPair(
                    panel_a=di.panel_id,
                    panel_b=dj.panel_id,
                    column_set_equal=column_set_equal,
                    n_cols_a=di.n_cols,
                    n_cols_b=dj.n_cols,
                    note=note,
                )
            )
    return tuple(pairs)


def build_panel_fingerprint(
    diagnostics: tuple[PerRowDiagnostic, ...],
) -> PanelFingerprint:
    return PanelFingerprint(
        n_rows_per_panel=_aggregate_n_rows(diagnostics),
        column_count_per_panel=_aggregate_column_count(diagnostics),
        non_null_fraction_per_column=_aggregate_non_null_fraction(diagnostics),
        date_min_per_panel=_aggregate_date_min(diagnostics),
        date_max_per_panel=_aggregate_date_max(diagnostics),
        cross_row_consistency_flag=_compute_cross_row_consistency(diagnostics),
    )


def fingerprint_to_jsonable(fp: PanelFingerprint) -> dict[str, object]:
    """Serialize PanelFingerprint to a json-deterministic plain dict."""
    return {
        "n_rows_per_panel": dict(fp.n_rows_per_panel),
        "column_count_per_panel": dict(fp.column_count_per_panel),
        "non_null_fraction_per_column": {
            k: dict(v) for k, v in fp.non_null_fraction_per_column.items()
        },
        "date_min_per_panel": dict(fp.date_min_per_panel),
        "date_max_per_panel": dict(fp.date_max_per_panel),
        "cross_row_consistency_flag": [
            {
                "panel_a": p.panel_a,
                "panel_b": p.panel_b,
                "column_set_equal": p.column_set_equal,
                "n_cols_a": p.n_cols_a,
                "n_cols_b": p.n_cols_b,
                "note": p.note,
            }
            for p in fp.cross_row_consistency_flag
        ],
    }


def assert_byte_exact_against_audit(
    fp: PanelFingerprint,
    audit: Mapping[str, Mapping[str, object]],
) -> None:
    """Hard assertion: panel-fingerprint overlap fields byte-exact vs `_audit_summary.json`."""
    for panel_id, n_rows in fp.n_rows_per_panel.items():
        audit_row = audit[panel_id]
        assert n_rows == int(audit_row["n_obs"]), (
            f"{panel_id}: n_rows={n_rows} != audit n_obs={audit_row['n_obs']}"
        )
        # date_min / date_max byte-exact check, with deferred-empty exception.
        audit_dt_min = audit_row.get("dt_min")
        audit_dt_max = audit_row.get("dt_max")
        fp_dt_min = fp.date_min_per_panel[panel_id]
        fp_dt_max = fp.date_max_per_panel[panel_id]
        if n_rows == 0:
            # Deferred-empty panels: both fingerprint and audit carry None.
            assert fp_dt_min is None and audit_dt_min is None, (
                f"{panel_id} deferred-empty dt_min: fp={fp_dt_min}, audit={audit_dt_min}"
            )
            assert fp_dt_max is None and audit_dt_max is None, (
                f"{panel_id} deferred-empty dt_max: fp={fp_dt_max}, audit={audit_dt_max}"
            )
        else:
            assert fp_dt_min == audit_dt_min, (
                f"{panel_id}: dt_min fp={fp_dt_min} != audit={audit_dt_min}"
            )
            assert fp_dt_max == audit_dt_max, (
                f"{panel_id}: dt_max fp={fp_dt_max} != audit={audit_dt_max}"
            )


def emit_fingerprint_json(payload: dict[str, object], out_path: Path) -> str:
    """Write the fingerprint JSON deterministically; return SHA-256 of the file."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    rendered = json.dumps(payload, indent=2, sort_keys=True)
    out_path.write_text(rendered)
    h = hashlib.sha256()
    with out_path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


# ---- Execute trio 2 ----

_AUDIT = _load_audit(_DATA_DIR_NB1_S1)
_FP = build_panel_fingerprint(_DIAGNOSTICS)

# Hard assertion vs upstream Phase 5a authority (n_obs / dt_min / dt_max).
assert_byte_exact_against_audit(_FP, _AUDIT)

# Cross-row consistency: among non-deferred rows, the column-set-equal flag must hold
# pairwise EXCEPT where Row 6 (parsimonious controls) participates — Row 6 intentionally
# carries a column-set subset (parsimonious 3-control) per spec §10 Row 6.
_NON_DEFERRED_EXPECTED_EQ = True  # for non-Row-6, non-deferred pairs
_ROW_6 = "row_06_parsimonious_controls"
_n_pairs = len(_FP.cross_row_consistency_flag)
assert _n_pairs == (14 * 13) // 2, f"Expected 91 pairs, got {_n_pairs}"

_pair_summary = {"true": 0, "false": 0, "na": 0}
for pair in _FP.cross_row_consistency_flag:
    if pair.column_set_equal is True:
        _pair_summary["true"] += 1
    elif pair.column_set_equal is False:
        _pair_summary["false"] += 1
    else:
        _pair_summary["na"] += 1

# Sanity: deferred-empty pairs must total C(2,1)*12 + C(2,2) = 24 + 1 = 25 pairs.
# (Row 9 with 12 non-deferred + Row 10 with 12 non-deferred + Row9-Row10 = 25.)
assert _pair_summary["na"] == 25, f"Expected 25 N/A pairs, got {_pair_summary['na']}"

# Among non-deferred non-Row-6 pairs, all must be column_set_equal=True.
# Among pairs involving Row 6 (parsimonious), expect column_set_equal=False (subset).
_row6_pairs_false = sum(
    1 for p in _FP.cross_row_consistency_flag
    if (p.panel_a == _ROW_6 or p.panel_b == _ROW_6)
    and p.column_set_equal is False
)
# Row 6 vs each of the other 11 non-deferred rows = 11 false pairs.
assert _row6_pairs_false == 11, f"Expected 11 Row-6 column-subset false pairs, got {_row6_pairs_false}"
# All other false pairs must be zero (i.e., total false = 11).
assert _pair_summary["false"] == 11, (
    f"Unexpected column-set drift: total false pairs = {_pair_summary['false']}, expected 11 (Row-6 only)"
)

# Emit the fingerprint JSON.
_OUT_DIR = env._CONTRACTS_DIR / "notebooks" / "abrigo_y3_x_d" / "estimates"
_OUT_PATH = _OUT_DIR / "panel_fingerprint.json"
_PAYLOAD = fingerprint_to_jsonable(_FP)
_FP_SHA = emit_fingerprint_json(_PAYLOAD, _OUT_PATH)

# Render summary.
print(f"panel_fingerprint.json emitted at: {_OUT_PATH}")
print(f"panel_fingerprint.json SHA-256:    {_FP_SHA}")
print()
print("Fingerprint required-field coverage:")
print(f"  n_rows_per_panel:               {len(_FP.n_rows_per_panel)} panels")
print(f"  column_count_per_panel:         {len(_FP.column_count_per_panel)} panels")
print(f"  non_null_fraction_per_column:   {len(_FP.non_null_fraction_per_column)} panels")
print(f"  date_min_per_panel:             {len(_FP.date_min_per_panel)} panels")
print(f"  date_max_per_panel:             {len(_FP.date_max_per_panel)} panels")
print(f"  cross_row_consistency_flag:     {len(_FP.cross_row_consistency_flag)} pairs")
print()
print("Cross-row consistency flag distribution:")
print(f"  column_set_equal = True:   {_pair_summary['true']} pairs (uniform-schema rows)")
print(f"  column_set_equal = False:  {_pair_summary['false']} pairs (Row-6 parsimonious-subset by spec)")
print(f"  column_set_equal = N/A:    {_pair_summary['na']} pairs (deferred-empty Rows 9 / 10)")
print()
print("Byte-exact reproduction vs `_audit_summary.json`: PASS (n_rows / dt_min / dt_max all 14 panels)")


panel_fingerprint.json emitted at: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/notebooks/abrigo_y3_x_d/estimates/panel_fingerprint.json
panel_fingerprint.json SHA-256:    c8730752a561dd5d6c9b01ffefe4b6db97053506940e4fadf86346ca2fbce779

Fingerprint required-field coverage:
  n_rows_per_panel:               14 panels
  column_count_per_panel:         14 panels
  non_null_fraction_per_column:   14 panels
  date_min_per_panel:             14 panels
  date_max_per_panel:             14 panels
  cross_row_consistency_flag:     91 pairs

Cross-row consistency flag distribution:
  column_set_equal = True:   55 pairs (uniform-schema rows)
  column_set_equal = False:  11 pairs (Row-6 parsimonious-subset by spec)
  column_set_equal = N/A:    25 pairs (deferred-empty Rows 9 / 10)

Byte-exact reproduction vs `_audit_summary.json`: PASS (n_rows / dt_min / dt_max all 14 panels)


### Interpretation

**Cross-row consistency aggregation establishes structural uniformity of the 14 Rev-2 panels.** All 91 ordered pairs (C(14,2)) of the 14 panels are evaluated against the column-set-equality rule: 55 pairs return `column_set_equal = True` (the 12 non-deferred rows excluding Row 6 form a structurally uniform set, all sharing the same `(week_start, y3_value, x_d, vix_avg, oil_return, us_cpi_surprise, banrep_rate_surprise, fed_funds_weekly, intervention_dummy, copm_diff, brl_diff, eur_diff, kes_diff)` column-set; ordered pairs among them = C(11,2)=55); 11 pairs return `column_set_equal = False` (Row 6 parsimonious-controls vs each of the other 11 non-deferred rows — Row 6 intentionally drops `us_cpi_surprise`, `banrep_rate_surprise`, `fed_funds_weekly` per spec §10 Row 6, yielding a 10-column schema vs the 13-column primary schema); and 25 pairs return `column_set_equal = N/A` (any pair involving Row 9 Y₃-bond-diagnostic or Row 10 population-weighted, both deferred-empty per spec §10 ε.2 / ε.3 — 12 pairs each Row 9 / Row 10 against non-deferred + 1 Row-9-vs-Row-10 = 25). The structural uniformity of the 14-row sensitivity ladder is therefore confirmed: refit-at-fit-time transformations in NB3 (Row 5 lag, Row 11 Student-t, Row 12 HAC(12), Row 13 first-differenced, Row 14 WC-CPI-weights-sensitivity) operate on identical input column-sets, and Row 6's parsimonious-controls subset is a published-spec deviation, not silent drift.

**Panel-fingerprint JSON emitted byte-exact against the upstream Phase 5a contract.** The `panel_fingerprint.json` file at `notebooks/abrigo_y3_x_d/estimates/panel_fingerprint.json` carries the 6 required fields (`n_rows_per_panel`, `column_count_per_panel`, `non_null_fraction_per_column`, `date_min_per_panel`, `date_max_per_panel`, `cross_row_consistency_flag`) with byte-exact reproduction of the `_audit_summary.json` overlap fields (n_rows / dt_min / dt_max for all 14 panels — hard-asserted via `assert_byte_exact_against_audit`). The deterministic JSON serialization (`json.dumps(payload, indent=2, sort_keys=True)`) produces a stable file SHA-256 that downstream consumers (NB2 §1, NB3) can pin. The Rev-5.3.5 Minteo-fintech scope-mismatch close-out interpretation is preserved: the fingerprint locks the byte-exact contract on the Rev-2 panels (which were measured against the Minteo-fintech X_d at `0xc92e8fc2...`), and any future β-track Mento-native panel (Task 11.P.spec-β / Task 11.P.exec-β) must replace this fingerprint structure cell-for-cell against the canonical Mento-native COPm address `0x8A567e2aE79CA692Bd748aB832081C45de4041eA` per the Rev-5.3.5 β-disposition memo.

**Forward-pointer.** NB1 §2 (Y₃ component decomposition — per-country diff-component validation against the equal-weight aggregator) follows in NB-α sub-task 3. NB1 §1 closes here.

## §2 — Y₃ component decomposition (per-country; equal-weight cross-country aggregation)

### Why-markdown (4-part citation block)

**Reference.** `project_y3_inequality_differential_design` (4-country panel CO/BR/KE/EU; equal-weight cross-country aggregation; pre-registered 60/25/15 WC-CPI within-country weights); `project_abrigo_inequality_hedge_thesis` (rich-asset returns vs. working-class consumption returns differential; Minteo-fintech scope-mismatch close-out under Rev-2); `contracts/docs/superpowers/specs/2026-04-24-y3-inequality-differential-design.md` (the immutable design doc); Rev-2 spec at `contracts/.scratch/2026-04-25-task110-rev2-spec-A-autonomous.md` §1.3 + §1.4 (outcome-variable definition + identity-transform pre-registration); Phase 5a manifest at `contracts/.scratch/2026-04-25-task110-rev2-data/manifest.md` for the 3-country composition under `source_methodology = y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` (KE skipped because the KNBS WC-CPI series forced source-side exclusion in the Phase 5a build).

**Why used.** Y₃ is the inequality-differential outcome variable. The component decomposition validates that the Y₃ aggregate column on every Rev-2 panel parquet is exactly the equal-weight average of the three per-country differentials `(copm_diff, brl_diff, eur_diff)` — KE is structurally NULL on the 3-country panel by `_ke_skip_3country_ke_unavailable` design. The within-country WC-CPI weights (60% food / 25% energy+housing-utilities / 15% transport-fuel; World Bank LAC bottom-quintile basket per the consumption-proxy-research §4) are pre-registered and traced here for provenance — they enter at the source-table level (`onchain_y3_weekly` upstream) and are NOT re-computed at the panel stage. This trio's job is to verify the identity `y3_value ≡ (copm_diff + brl_diff + eur_diff) / 3` byte-exact, NOT to re-weight CPI components.

**Relevance to results.** Byte-exact reproduction of `y3_value` from per-country components is the necessary condition for byte-exact regression-coefficient reproduction in NB2 §1 (the OLS+HAC(4) primary on `panel_row_01_primary.parquet`). Any drift here would surface a panel-construction bug that would silently propagate into every β̂ estimate. The Rev-2 closes scope-mismatch on Minteo-fintech X_d by holding the inequality-differential Y₃ unchanged across all 14 panels — so this identity check applies uniformly to every parquet under `source_methodology = y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` (Rows 1–3, 5–8, 11–14).

**Connection to simulator.** The Abrigo simulator's payoff structure is calibrated against the Y₃ inequality-differential surface. Component-level traceability lets future Rev-3 / β-track Minteo-fintech X_d panels (and any future Rev-3 ζ-group convex-payoff extensions covering quantile / GARCH-X / lower-tail / option-implied vol) plug into the same simulator interface without re-deriving the differential — the per-country components remain the canonical inputs, only the aggregation weights or country-membership change. Pre-registered 60/25/15 WC-CPI weights become an audit anchor against any silent re-weighting between Rev-2 and Rev-3.


In [4]:
"""NB1 §2 — Y₃ component decomposition (per-country; equal-weight cross-country aggregation).

Validates the identity `y3_value ≡ (copm_diff + brl_diff + eur_diff) / 3` byte-exact
on `panel_row_01_primary.parquet` under `source_methodology =
y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` (KE skipped per
3-country panel design). Documents pre-registered within-country WC-CPI weights
(60/25/15 food/energy+housing-utilities/transport-fuel; World Bank LAC bottom-quintile
basket) as an audit anchor against any silent re-weighting. Functional-Python:
frozen dataclasses, pure free functions, full typing, no inheritance.
"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from types import MappingProxyType
from typing import Mapping

import duckdb
import pandas as pd


# ---- Frozen-dataclass result records ----

@dataclass(frozen=True, slots=True)
class WCCPIWeights:
    """Pre-registered within-country WC-CPI component weights.

    Source: 2026-04-24-y3-consumption-proxy-research.md §4 lit-grounded weights
    (World Bank LAC bottom-quintile basket). These weights enter at the
    source-table layer (`onchain_y3_weekly` upstream); the panel stage
    consumes already-weighted Δlog(WC_CPI) per country.
    """
    food: float
    energy_housing: float
    transport_fuel: float

    def total(self) -> float:
        return self.food + self.energy_housing + self.transport_fuel


@dataclass(frozen=True, slots=True)
class Y3IdentityCheck:
    """Byte-exact identity check result for Y₃ aggregate vs. per-country sum/3."""
    n_obs: int
    max_abs_diff: float
    mean_abs_diff: float
    tolerance: float
    passed: bool
    countries_in_aggregate: tuple[str, ...]
    kes_null_count: int
    aggregation_weight_each: float


# ---- Pure helper functions ----

def _panel_path(data_dir: Path, panel_id: str) -> Path:
    return data_dir / f"panel_{panel_id}.parquet"


def load_y3_components_frame(
    parquet_path: Path,
) -> pd.DataFrame:
    """Load week_start + Y₃ aggregate + per-country differentials.

    Returns a DataFrame with columns:
      week_start, y3_value, copm_diff, brl_diff, kes_diff, eur_diff
    """
    conn = duckdb.connect()
    try:
        df = conn.sql(
            f"""
            SELECT
                week_start,
                y3_value,
                copm_diff,
                brl_diff,
                kes_diff,
                eur_diff
            FROM '{parquet_path}'
            ORDER BY week_start
            """
        ).df()
    finally:
        conn.close()
    return df


def compute_equal_weight_aggregate(
    df: pd.DataFrame,
    components: tuple[str, ...] = ("copm_diff", "brl_diff", "eur_diff"),
) -> pd.Series:
    """Reconstruct Y₃ aggregate as equal-weight mean over the named components.

    For the 3-country panel under `_ke_skip_3country_ke_unavailable`, the
    canonical components are (copm_diff, brl_diff, eur_diff); KE is NULL
    by design and excluded from the aggregation.
    """
    return df[list(components)].sum(axis=1) / float(len(components))


def check_y3_identity(
    df: pd.DataFrame,
    *,
    components: tuple[str, ...] = ("copm_diff", "brl_diff", "eur_diff"),
    tolerance: float = 1e-12,
) -> Y3IdentityCheck:
    reconstructed = compute_equal_weight_aggregate(df, components)
    abs_diff = (df["y3_value"] - reconstructed).abs()
    max_abs = float(abs_diff.max())
    mean_abs = float(abs_diff.mean())
    kes_null_count = int(df["kes_diff"].isna().sum())
    return Y3IdentityCheck(
        n_obs=int(len(df)),
        max_abs_diff=max_abs,
        mean_abs_diff=mean_abs,
        tolerance=tolerance,
        passed=max_abs <= tolerance,
        countries_in_aggregate=components,
        kes_null_count=kes_null_count,
        aggregation_weight_each=1.0 / float(len(components)),
    )


def summarize_per_country_components(df: pd.DataFrame) -> pd.DataFrame:
    """Return a per-component summary-statistics frame (CO/BR/KE/EU)."""
    rows = []
    for col, country in (
        ("copm_diff", "Colombia (CO; COPM)"),
        ("brl_diff", "Brazil (BR; BRLm)"),
        ("kes_diff", "Kenya (KE; KESm) — SKIPPED on 3-country panel"),
        ("eur_diff", "Eurozone (EU; EURm)"),
    ):
        s = df[col]
        rows.append(
            {
                "component": col,
                "country": country,
                "n_non_null": int(s.notna().sum()),
                "n_null": int(s.isna().sum()),
                "mean": float(s.mean()) if s.notna().any() else float("nan"),
                "std": float(s.std()) if s.notna().any() else float("nan"),
                "min": float(s.min()) if s.notna().any() else float("nan"),
                "max": float(s.max()) if s.notna().any() else float("nan"),
            }
        )
    return pd.DataFrame(rows)


# ---- Pre-registered constants (immutable provenance) ----

WC_CPI_WEIGHTS: WCCPIWeights = WCCPIWeights(
    food=0.60,
    energy_housing=0.25,
    transport_fuel=0.15,
)
WC_CPI_WEIGHT_PROVENANCE: Mapping[str, str] = MappingProxyType(
    {
        "source_doc": (
            "contracts/docs/superpowers/specs/"
            "2026-04-24-y3-inequality-differential-design.md §4"
        ),
        "consumption_proxy_research": (
            "2026-04-24-y3-consumption-proxy-research.md §4 "
            "(lit-grounded weights)"
        ),
        "basket_origin": "World Bank LAC bottom-quintile budget shares",
        "memory_anchor": "project_y3_inequality_differential_design",
        "stage_applied": (
            "source-table layer (onchain_y3_weekly upstream); "
            "NOT re-computed at panel stage"
        ),
    }
)

CROSS_COUNTRY_AGGREGATION: Mapping[str, str] = MappingProxyType(
    {
        "method": "equal-weight",
        "weight_each": "1/3",
        "components": "(copm_diff, brl_diff, eur_diff)",
        "ke_status": (
            "structurally NULL on 3-country panel under "
            "source_methodology=y3_v2_co_dane_br_bcb_eu_eurostat_"
            "ke_skip_3country_ke_unavailable"
        ),
    }
)


# ---- Driver ----

DATA_DIR: Path = Path("../../.scratch/2026-04-25-task110-rev2-data")
PRIMARY_PANEL_ID: str = "row_01_primary"
TOLERANCE: float = 1e-12

primary_path = _panel_path(DATA_DIR, PRIMARY_PANEL_ID)
y3_frame = load_y3_components_frame(primary_path)

# Identity check: y3_value ≡ (copm_diff + brl_diff + eur_diff) / 3
identity_result = check_y3_identity(
    y3_frame,
    components=("copm_diff", "brl_diff", "eur_diff"),
    tolerance=TOLERANCE,
)

# Hard assertion: any drift would falsify the panel-construction step
assert identity_result.passed, (
    f"Y₃ identity FAILED: max_abs_diff={identity_result.max_abs_diff:.3e} "
    f"> tolerance={TOLERANCE:.0e}. Panel-construction bug — investigate "
    "before proceeding to NB2 estimation."
)
assert identity_result.kes_null_count == identity_result.n_obs, (
    f"KE expected fully NULL on 3-country panel; got "
    f"kes_null_count={identity_result.kes_null_count} of n_obs="
    f"{identity_result.n_obs}."
)

# Render: identity-check summary
print("=== Y₃ identity check (panel_row_01_primary) ===")
print(f"  n_obs                      : {identity_result.n_obs}")
print(f"  components                 : {identity_result.countries_in_aggregate}")
print(f"  weight per component       : {identity_result.aggregation_weight_each:.6f}")
print(f"  max |y3_value − recon|     : {identity_result.max_abs_diff:.3e}")
print(f"  mean |y3_value − recon|    : {identity_result.mean_abs_diff:.3e}")
print(f"  tolerance                  : {identity_result.tolerance:.0e}")
print(f"  identity passed            : {identity_result.passed}")
print(f"  kes_diff NULL count / n    : {identity_result.kes_null_count}/{identity_result.n_obs}")
print()

# Render: per-country component summary
component_summary = summarize_per_country_components(y3_frame)
print("=== Per-country differential summary (76-week primary panel) ===")
print(component_summary.to_string(index=False))
print()

# Render: pre-registered WC-CPI weights traceability
print("=== Pre-registered within-country WC-CPI weights (60/25/15) ===")
print(f"  food                           : {WC_CPI_WEIGHTS.food:.2f}")
print(f"  energy + housing-utilities     : {WC_CPI_WEIGHTS.energy_housing:.2f}")
print(f"  transport-fuel                 : {WC_CPI_WEIGHTS.transport_fuel:.2f}")
print(f"  total (sanity check)           : {WC_CPI_WEIGHTS.total():.2f}")
print()
print("Provenance:")
for k, v in WC_CPI_WEIGHT_PROVENANCE.items():
    print(f"  {k:30s} : {v}")
print()

# Render: cross-country aggregation rule
print("=== Cross-country aggregation (equal-weight 1/3 per country; KE skipped) ===")
for k, v in CROSS_COUNTRY_AGGREGATION.items():
    print(f"  {k:14s} : {v}")
print()

# Render: head/tail of per-week reconstruction (compact)
recon = compute_equal_weight_aggregate(
    y3_frame, ("copm_diff", "brl_diff", "eur_diff")
)
recon_view = pd.DataFrame(
    {
        "week_start": y3_frame["week_start"],
        "copm_diff": y3_frame["copm_diff"],
        "brl_diff": y3_frame["brl_diff"],
        "eur_diff": y3_frame["eur_diff"],
        "y3_value": y3_frame["y3_value"],
        "reconstructed": recon,
        "abs_diff": (y3_frame["y3_value"] - recon).abs(),
    }
)
print("=== Per-week reconstruction (head 5 + tail 5 of 76 weeks) ===")
print(pd.concat([recon_view.head(5), recon_view.tail(5)]).to_string(index=False))


=== Y₃ identity check (panel_row_01_primary) ===
  n_obs                      : 76
  components                 : ('copm_diff', 'brl_diff', 'eur_diff')
  weight per component       : 0.333333
  max |y3_value − recon|     : 0.000e+00
  mean |y3_value − recon|    : 0.000e+00
  tolerance                  : 1e-12
  identity passed            : True
  kes_diff NULL count / n    : 76/76

=== Per-country differential summary (76-week primary panel) ===
component                                       country  n_non_null  n_null     mean      std       min      max
copm_diff                           Colombia (CO; COPM)          76       0 0.007515 0.022665 -0.055248 0.066695
 brl_diff                             Brazil (BR; BRLm)          76       0 0.005165 0.020314 -0.042435 0.081865
 kes_diff Kenya (KE; KESm) — SKIPPED on 3-country panel           0      76      NaN      NaN       NaN      NaN
 eur_diff                           Eurozone (EU; EURm)          76       0 0.002119 0.018759 -0.0

### Interpretation

The Y₃ component-decomposition identity holds **byte-exact** on the primary
panel: `max |y3_value − (copm_diff + brl_diff + eur_diff) / 3| = 0.0` over all
n = 76 weekly observations (tolerance threshold 1e-12). The identity check
passes the necessary condition for byte-exact regression-coefficient
reproduction in NB2 §1; any future drift here would surface a panel-construction
bug at the `onchain_y3_weekly` source-table layer before β̂ estimates ever
incorporate it.

**Cross-country aggregation is equal-weight** (1/3 per country) over the
canonical 3-country composition `(copm_diff, brl_diff, eur_diff)`. KE
(`kes_diff`) is structurally NULL on every row of the 3-country panel —
76/76 NULL count — by `source_methodology =
y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` design.
The 4-country `(1/4) × (Δ_CO + Δ_BR + Δ_KE + Δ_EU)` form in the immutable
design doc collapses to the 3-country `(1/3) × (Δ_CO + Δ_BR + Δ_EU)` form
in this Phase 5a build because KNBS WC-CPI series forced source-side exclusion;
the design-doc form remains the canonical statement and is recovered if KE
WC-CPI re-enters at a future Rev-3 build.

**Within-country WC-CPI weights are the pre-registered 60/25/15 split**
(60% food / 25% energy+housing-utilities / 15% transport-fuel; World Bank LAC
bottom-quintile basket per the consumption-proxy-research §4). These weights
are an *audit anchor* — they enter at the source-table layer (`onchain_y3_weekly`
upstream) and are NOT re-computed at the panel stage. Row 14
(`panel_row_14_wc_cpi_weights_sens.parquet`) is the formal sensitivity
parquet that re-weights at fit time using `(copm_diff, brl_diff, eur_diff)`
as inputs; it is NOT exercised in this trio (Row 14 sensitivity is an NB2
fit-time concern, not an EDA-stage concern).

The Rev-2 closes scope-mismatch on Minteo-fintech X_d by holding Y₃ unchanged
across all 14 panels — so this identity is uniform across Rows 1–3, 5–8,
11–14. (Rows 4 and 9 use alternative `source_methodology` literals; they
are out of scope for this primary-panel identity check, which targets the
Rev-2 primary spec only.)

**Forward-pointer.** NB1 §3 (X_d distribution + diurnal pattern) follows in
sub-task 4 of the NB-α dispatch sequence.
